In [1]:
import mlflow
mlflow.set_tracking_uri("http://127.0.0.1:5000")
mlflow.set_experiment("MLflow Quickstart")



/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


<Experiment: artifact_location='mlflow-artifacts:/2', creation_time=1774957102775, experiment_id='2', last_update_time=1774957102775, lifecycle_stage='active', name='MLflow Quickstart', tags={}, workspace='default'>

In [11]:
import pandas as pd
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "random_state": 8888,
}

In [7]:
# Enable autologging for scikit-learn
mlflow.sklearn.autolog()

# Just train the model normally
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

2026/03/31 15:23:29 WARNING mlflow.utils.autologging_utils: Encountered unexpected error during sklearn autologging: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,8888
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [9]:
with mlflow.start_run():
    mlflow.log_params(params)

    model = LogisticRegression(**params)
    model.fit(X_train, y_train)

    model_info = mlflow.sklearn.log_model(sk_model=model, name="iris_model")

    y_pred = model.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    mlflow.log_metric("accuracy", accuracy)

    mlflow.set_tag("Training info", "Quickstart demo")


MlflowException: API request to endpoint /api/2.0/mlflow/runs/create failed with error code 403 != 200. Response body: ''

In [9]:
mlflow.set_experiment(experiment_name="Tuning RF-Regressor")

from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_california_housing

X, y = fetch_california_housing(return_X_y=True)
X_train, X_val, y_train, y_val = train_test_split(X, y, random_state=0)

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error

def objective(trial):
    with mlflow.start_run(nested=True, run_name=f"trial_{trial.number}") as child_run:

        rf_max_depth = trial.suggest_int("rf_max_depth", 3, 10, )
        rf_n_estimators = trial.suggest_int("rf_n_estimators", 3, 10)

        params = {
            "max_depth": rf_max_depth,
            "n_estimators": rf_n_estimators,
        }
        mlflow.log_params(params)

        model = RandomForestRegressor(**params)
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        error = mean_squared_error(y_val, y_pred)
        
        mlflow.log_metric(key="MSE", value=error)
        mlflow.sklearn.log_model(model, name="RandomForestRegressor")

        trial.set_user_attr("run_id", child_run.info.run_id)

        return error

import optuna
with mlflow.start_run(run_name="study") as run:
    n_trials = 30
    mlflow.log_param("n_trials", n_trials)

    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=n_trials)

    mlflow.log_params(study.best_trial.params)
    mlflow.log_metrics({"best_error": study.best_value})
    if best_run_id := study.best_trial.user_attrs.get("run_id"):
        mlflow.log_param("best_child_run_id", best_run_id)

[I 2026-03-31 15:45:09,454] A new study created in memory with name: no-name-9ab16dca-db64-418a-bd2e-f2165b3d3836
2026/03/31 15:45:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-03-31 15:45:14,388] Trial 0 finished with value: 0.34892235516778963 and parameters: {'rf_max_depth': 9, 'rf_n_estimators': 8}. Best is trial 0 with value: 0.34892235516778963.


🏃 View run trial_0 at: http://127.0.0.1:5000/#/experiments/3/runs/08c6cc8eed5d4833a4dce1209f589a9e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/31 15:45:14 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-03-31 15:45:18,966] Trial 1 finished with value: 0.3915560444245105 and parameters: {'rf_max_depth': 8, 'rf_n_estimators': 4}. Best is trial 0 with value: 0.34892235516778963.


🏃 View run trial_1 at: http://127.0.0.1:5000/#/experiments/3/runs/ad5f689a43f94a5ea8742e89b351d28a
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/31 15:45:19 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-03-31 15:45:23,270] Trial 2 finished with value: 0.5504915943164075 and parameters: {'rf_max_depth': 4, 'rf_n_estimators': 9}. Best is trial 0 with value: 0.34892235516778963.


...

🏃 View run trial_27 at: http://127.0.0.1:5000/#/experiments/3/runs/c65b529de5ec4a0d988fff2850a5756c
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/31 15:47:36 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-03-31 15:47:42,420] Trial 28 finished with value: 0.4034708078929521 and parameters: {'rf_max_depth': 7, 'rf_n_estimators': 10}. Best is trial 21 with value: 0.322694236361921.


🏃 View run trial_28 at: http://127.0.0.1:5000/#/experiments/3/runs/9edd0838a2074ae781d558c64956cb23
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


2026/03/31 15:47:43 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
[I 2026-03-31 15:47:50,272] Trial 29 finished with value: 0.3460302114263926 and parameters: {'rf_max_depth': 9, 'rf_n_estimators': 8}. Best is trial 21 with value: 0.322694236361921.


🏃 View run trial_29 at: http://127.0.0.1:5000/#/experiments/3/runs/5816f068ac7148f085e9acb275b44901
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3
🏃 View run study at: http://127.0.0.1:5000/#/experiments/3/runs/c37b94590c5b4d18863332e209137677
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/3


In [46]:
import mlflow
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

# Enable autologging
mlflow.set_experiment("Deep learning demo")

# Create synthetic data
X = torch.randn(1000, 784)
y = torch.randint(0, 10, (1000,))
train_loader = DataLoader(TensorDataset(X, y), batch_size=32, shuffle=True)
model = nn.Sequential(nn.Linear(784, params["hidden_channel"]), nn.ReLU(), nn.Linear(params["hidden_channel"], 10))
optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
criterion = params["loss_fn"]

params = {
    "hidden_channel": 128,
    "lr": 0.001,
    "loss_fn": nn.CrossEntropyLoss(),
    "epochs": 1000,
    "model_arch": model,
    "optimizer": optimizer
}

mlflow.enable_system_metrics_logging()
with mlflow.start_run(run_name="training"):

    # Your existing PyTorch code works unchanged
    mlflow.log_params(params)

    # forward
    for epoch in range(params["epochs"]):

        train_loss = 0
        train_accuracy = 0
        total = 0

        model.train()
        for batch in train_loader:
            X_train, y_train = batch
            optimizer.zero_grad()
            out = model(X_train)
            loss = criterion(out, y_train)
            loss.backward()
            optimizer.step()

            train_loss += loss * X_train.shape[0]
            train_accuracy += (out.detach().argmax(dim=1) == y_train).float().mean().item() * X_train.shape[0]
            total += X_train.shape[0]

        train_loss /= total
        train_accuracy /= total

        mlflow.log_metrics({"train_loss": train_loss, "train_accuracy": train_accuracy}, step=epoch)
    signature = mlflow.models.infer_signature(X_train.numpy(), model(X_train).detach().numpy())
    mlflow.pytorch.log_model(pytorch_model=model, name="MLP", signature=signature)

2026/03/31 17:13:51 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/31 17:13:51 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/31 17:14:54 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
2026/03/31 17:14:57 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/03/31 17:14:57 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!


🏃 View run training at: http://127.0.0.1:5000/#/experiments/4/runs/622925e2948a4521a18dd020b64c5fc8
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [ ]:
from sklearn.datasets import load_iris

iris = load_iris()
iris_df = pd.DataFrame(data=iris.data, columns=iris.feature_names)
iris_df["target"] = iris.target

import pandas as pd

def prepare_data(df: pd.DataFrame):
    X = torch.tensor(df.iloc[:, :-1].values, dtype=torch.float32)
    y = torch.tensor(df.iloc[:, -1].values, dtype=torch.long)
    return X, y

def compute_accuracy(out: torch.Tensor, y_true: torch.Tensor):
    return (out.argmax(dim=1) == y_true).float().mean()

import torch
import torch.nn as nn

params = {
    "model": model,
    "in_channel": iris_df.values[:, :-1].shape[1],
    "hidden_channel": 16,
    "out_channel": len(iris.target_names),
    "dataset-name": "iris",
    "test_size": 0.2,
    "random_state": 42,
    "lr": 0.001,
    "criterion": "nn.CrossEntropyLoss()",
    "optimizer": "Adam",
    "epochs": 100
}

class IrisClassifier(nn.Module):
    def __init__(self, input_channel: int, hidden_channel: int, out_channel: int):
        super().__init__()

        self.linear = nn.Sequential(
            nn.Linear(input_channel, hidden_channel),
            nn.ReLU(),
            nn.Linear(hidden_channel, out_channel)
        )
    
    def forward(self, x: torch.Tensor):
        return self.linear(x)

train_df, test_df = train_test_split(iris_df, test_size=params["test_size"], random_state=params["random_state"])
train_dataset = mlflow.data.from_pandas(train_df, name="iris-train-df")

X_train, y_train = prepare_data(train_df)
X_test, y_test = prepare_data(test_df)


model = IrisClassifier(params["in_channel"], params["hidden_channel"], params["out_channel"])

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(params=model.parameters(), lr=params["lr"])

with mlflow.start_run(run_name="logging-checkpoints") as run:
    mlflow.log_params(params)

    model.train()
    for epoch in range(params["epochs"]):
        optimizer.zero_grad()
        out = model(X_train)
        loss = criterion(out, y_train)
        loss.backward()
        optimizer.step()

        if epoch % 10 == 0:
            model_info = mlflow.pytorch.log_model(
                pytorch_model=model,
                name=f"IrisClassifer-checkpoint-{epoch}",
                step=epoch,
                input_example=X_train[:5].numpy(),
            )
            accuracy = compute_accuracy(out, y_train)
            mlflow.log_metric("train_accuracy", accuracy, step=epoch, dataset=train_dataset, model_id=model_info.model_id)
        
ranked_checkpoints = mlflow.search_logged_models(
    filter_string=f"source_run_id='{run.info.run_id}'",
    order_by=[{"field_name": "metrics.train_accuracy", "ascending":False}],
    output_format="list"
)

model_uri = ranked_checkpoints[0].model_uri

model = mlflow.pytorch.load_model(model_uri)


2026/03/31 18:02:42 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/03/31 18:02:42 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/03/31 18:02:42 WARNING mlflow.pytorch: Saving pytorch model by Pickle or CloudPickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization.The recommended safe alternative is to set 'export_model' to True to save the pytorch model using the safe graph model format.
/Users/tungvuduc/opt/anaconda3/envs/dlbio_arm64/lib/python3.12/site-packages/mlflow/types/utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way t

🏃 View run logging-checkpoints at: http://127.0.0.1:5000/#/experiments/4/runs/61958c95603649f3bc2b80d8c14fabb7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/4


In [103]:
import mlflow
import optuna
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

# Create synthetic dataset for demonstration
input_size = 784  # e.g., flattened 28x28 images
output_size = 10  # e.g., 10 classes

X_train = torch.randn(1000, input_size)
y_train = torch.randint(0, output_size, (1000,))
X_val = torch.randn(200, input_size)
y_val = torch.randint(0, output_size, (200,))

train_loader = DataLoader(TensorDataset(X_train, y_train), batch_size=32, shuffle=True)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=32)


def train_val(epochs: int, model: IrisClassifier, optimizer: torch.optim, criterion: nn.CrossEntropyLoss, train_loader: DataLoader, val_loader: DataLoader):
    for _ in range(epochs):

        val_loss = 0

        model.train()
        for batch in train_loader:
            X_train, y_train = batch
            optimizer.zero_grad()
            out = model(X_train)
            loss = criterion(out, y_train)
            loss.backward()
            optimizer.step()

        with torch.no_grad():
            model.eval()
            for batch in val_loader:
                X_val, y_val = batch
                out = model(X_val)
                val_loss += criterion(out, y_val).item()
        
        return val_loss/len(val_loader)

def objective(trial):

    with mlflow.start_run(nested=True):

        params = {
            "lr": trial.suggest_float("lr", 0.001, 0.01, log=True),
            "hidden_size": trial.suggest_int("hidden_size", 32, 256),
            "dropout": trial.suggest_float("dropout", 0.1, 0.8)
        }

        mlflow.log_params(params)

        model = nn.Sequential(
            nn.Linear(input_size, params["hidden_size"]),
            nn.ReLU(),
            nn.Dropout(params["dropout"]),
            nn.Linear(params["hidden_size"], output_size),
        )

        optimizer = torch.optim.Adam(model.parameters(), lr=params["lr"])
        criterion = nn.CrossEntropyLoss()

        val_loss = train_val(100, model, optimizer, criterion, train_loader, val_loader)

        mlflow.log_metric("val_loss", val_loss)
        return val_loss

mlflow.set_experiment("TuningMLP")
with mlflow.start_run(run_name="Tuning-lr-hidden-dropout"):
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=50)

    # logging the best params
    mlflow.log_params({key: value for key, value in study.best_params.items()})
    mlflow.log_metric("best_val_loss", study.best_value)

2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
[I 2026-04-01 14:59:14,773] A new study created in memory with name: no-name-2096cebc-ae9e-46a0-adc3-be6274adef7f
2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:14 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:14,907] Trial 0 finished with value: 2.3187992572784424 and parameters: {'lr': 0.001039550193045073, 'hidden_size': 164, 'dropout': 0.5325112

🏃 View run delicate-worm-914 at: http://127.0.0.1:5000/#/experiments/5/runs/009dbfb30e6047c68bf29322cfd904c1
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run righteous-lark-712 at: http://127.0.0.1:5000/#/experiments/5/runs/1630bc1eb4f44df0ae82cf5b3c49b66b
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:15,157] Trial 2 finished with value: 2.3482637064797536 and parameters: {'lr': 0.0039225563119268565, 'hidden_size': 83, 'dropout': 0.7927400270260995}. Best is trial 1 with value: 2.304215567452567.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:15,298] Trial 3 finished with value: 2.558251074382

🏃 View run capable-fawn-426 at: http://127.0.0.1:5000/#/experiments/5/runs/7033f945d2ac4edba15758bac0d5e0c0
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run delicate-slug-990 at: http://127.0.0.1:5000/#/experiments/5/runs/c73cf468ca8648438b8564674b7cb478
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:15,467] Trial 4 finished with value: 2.3310039724622453 and parameters: {'lr': 0.00113405199440548, 'hidden_size': 216, 'dropout': 0.19855504940241747}. Best is trial 1 with value: 2.304215567452567.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:15 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:15,607] Trial 5 finished with value: 2.355285133634

...

🏃 View run upbeat-eel-199 at: http://127.0.0.1:5000/#/experiments/5/runs/aaad93671eda41a2875d510c34553726
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run adventurous-colt-571 at: http://127.0.0.1:5000/#/experiments/5/runs/d7544f604bf14458be28cb0f45ff12ba
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:20,606] Trial 46 finished with value: 2.3192265714917863 and parameters: {'lr': 0.0013353112991767039, 'hidden_size': 33, 'dropout': 0.3040084671429064}. Best is trial 1 with value: 2.304215567452567.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:20,709] Trial 47 finished with value: 2.3085330213

🏃 View run sassy-skink-423 at: http://127.0.0.1:5000/#/experiments/5/runs/d49e78d06e8b495e97be905dc93d50e7
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run amusing-dog-779 at: http://127.0.0.1:5000/#/experiments/5/runs/7939a7c33a9145f1af7c1961ff6daf47
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5


2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:20,826] Trial 48 finished with value: 2.376923527036394 and parameters: {'lr': 0.002304983992306296, 'hidden_size': 42, 'dropout': 0.46197031823482515}. Best is trial 1 with value: 2.304215567452567.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Skip logging GPU metrics. Set logger level to DEBUG for more details.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Started monitoring system metrics.
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Stopping system metrics monitoring...
2026/04/01 14:59:20 INFO mlflow.system_metrics.system_metrics_monitor: Successfully terminated system metrics monitoring!
[I 2026-04-01 14:59:20,958] Trial 49 finished with value: 2.35521428925

🏃 View run grandiose-fowl-820 at: http://127.0.0.1:5000/#/experiments/5/runs/0e2ebe4d4722414b9eeef6591157a23e
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run delightful-kite-815 at: http://127.0.0.1:5000/#/experiments/5/runs/59bc744f106f4421bf4677ec13feb9cc
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
🏃 View run Tuning-lr-hidden-dropout at: http://127.0.0.1:5000/#/experiments/5/runs/7865de1099f044b08e7c59a24c90bc30
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/5
